# Week 79, Day 4: Dependency security

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w079d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
versions = ["1.9.0", "1.10.0", "1.2.0"]
print(sorted(versions))

def key(v):
    return tuple(int(part) for part in v.split("."))
print(sorted(versions, key=key))
print("1.10.0" > "1.9.0", key("1.10.0") > key("1.9.0"))

In [ ]:
import re

PRE = {"a": 0, "b": 1, "rc": 2}

def key(text):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", text)
    release = [int(x) for x in m[1].split(".")]
    while len(release) > 1 and release[-1] == 0:
        release.pop()
    pre = (PRE[m[2]], int(m[3])) if m[2] else (3, 0)
    return (tuple(release), pre)

versions = ["2.0", "2.0rc1", "1.10", "1.9.3", "2.0a2", "2.0.0b1", "1.10.0"]
print(sorted(versions, key=key))
print(key("1.10") == key("1.10.0"))

> **Key idea.** Dependencies include transitive ones, and all run with your permissions (some at install time). Risks: known vulnerabilities (advisories with CVE, GHSA or PYSEC ids and affected ranges), typosquats, dependency confusion between private and public indexes, compromised maintainers or updates, abandoned packages, licences. PEP 440: compare release numbers as tuples without trailing zeros; a1 < b1 < rc1 < final. Specifiers: ==, !=, <, <=, >, >=, ~= (compatible release), comma-separated and all required. Lock every package exactly with hashes and install with --require-hashes; audit the lock against advisories (normalise names per PEP 503), take the smallest fixed version, prioritise by severity; update often with Dependabot or Renovate and CI. Before adding a package: need, real name, maintenance, its own dependencies, licence; use --index-url for private packages and claim internal names publicly.

### Exercise: Comparing versions

Write a class `Version(text)` for PEP 440 versions of the form `N(.N)*`, optionally followed by `a`, `b` or `rc` and a number (raise `ValueError(f"not a version: {text!r}")` otherwise; surrounding spaces are ignored). Give it `release` (a tuple of ints as written), `pre` (`None`, or a tuple like `("rc", 1)`), `str()` giving the version as written (without the spaces), and comparisons (`==`, `<`, `<=`, `>`, `>=`, and hashing consistent with `==`) using the rules: trailing zeros in the release don't matter, and pre-releases come before the final release, in the order `a`, `b`, `rc`. Use `functools.total_ordering`.

In [ ]:
import functools
import re

@functools.total_ordering
class Version:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"parts\"\n    v = Version(\" 2.0rc1 \")\n    assert v.release == (2, 0) and v.pre == (\"rc\", 1) and str(v) == \"2.0rc1\"\n    assert Version(\"1.10.0\").release == (1, 10, 0) and Version(\"3\").pre is None\n    for bad in (\"1.x\", \"v1.0\", \"1..2\", \"\", \"2.0beta1\", \"1.0-rc1\"):\n        try:\n            Version(bad)\n        except ValueError as e:\n            assert str(e) == f\"not a version: {bad!r}\"\n        else:\n            raise AssertionError(bad)\n\ndef test_order():\n    \"PEP 440 ordering\"\n    texts = [\"2.0\", \"2.0rc1\", \"1.10\", \"1.9.3\", \"2.0a2\", \"2.0.0b1\", \"1.10.0.1\", \"0.9\", \"2.0a10\"]\n    assert [str(v) for v in sorted(Version(t) for t in texts)] == [\"0.9\", \"1.9.3\", \"1.10\", \"1.10.0.1\", \"2.0a2\", \"2.0a10\", \"2.0.0b1\", \"2.0rc1\", \"2.0\"]\n    assert Version(\"1.0\") == Version(\"1.0.0\") == Version(\"1\") and hash(Version(\"1.0\")) == hash(Version(\"1\"))\n    assert Version(\"1.0rc1\") != Version(\"1.0\") and Version(\"1.0b1\") < Version(\"1.0rc1\") <= Version(\"1.0rc1\")\n    assert len({Version(\"2\"), Version(\"2.0\"), Version(\"2.0.0\")}) == 1")

### Exercise: Does it satisfy the specifier?

Using `Version` (the starter includes it), write `satisfies(version, spec)`: `True` if the version text meets every comma-separated condition in `spec` (spaces anywhere are ignored; an empty spec allows anything). Operators: `==`, `!=`, `<`, `<=`, `>`, `>=`, and `~=V`, which means `>=V` and the same release numbers as V **except its last one** (so `~=2.28` needs `2.*`, and `~=2.28.1` needs `2.28.*`; `~=` needs at least two release numbers, else raise `ValueError("~= needs at least two release numbers")`). An unknown operator raises `ValueError(f"bad specifier: {part!r}")`. Then `best(available, spec)`: the highest version text in `available` that satisfies the spec and isn't a pre-release, or `None`.

In [ ]:
import functools
import re

PRE_ORDER = {"a": 0, "b": 1, "rc": 2}

@functools.total_ordering
class Version:
    def __init__(self, text):
        text = text.strip()
        m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", text)
        if not m:
            raise ValueError(f"not a version: {text!r}")
        self.text = text
        self.release = tuple(int(x) for x in m[1].split("."))
        self.pre = (m[2], int(m[3])) if m[2] else None
    def _key(self):
        release = list(self.release)
        while len(release) > 1 and release[-1] == 0:
            release.pop()
        return tuple(release), (PRE_ORDER[self.pre[0]], self.pre[1]) if self.pre else (3, 0)
    def __str__(self):
        return self.text
    def __eq__(self, other):
        return self._key() == other._key()
    def __lt__(self, other):
        return self._key() < other._key()
    def __hash__(self):
        return hash(self._key())

def satisfies(version, spec):
    ___

def best(available, spec):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_satisfies():\n    \"each operator\"\n    assert satisfies(\"2.31.0\", \">=2.28, <3\") and not satisfies(\"3.0\", \">=2.28,<3\") and satisfies(\"2.28\", \">=2.28.0\")\n    assert satisfies(\"1.4.2\", \"==1.4.2\") and satisfies(\"1.4.2\", \"==1.4.2.0\") and not satisfies(\"1.4.2\", \"!=1.4.2\")\n    assert satisfies(\"2.30\", \"~=2.28\") and not satisfies(\"3.1\", \"~=2.28\") and not satisfies(\"2.27\", \"~=2.28\")\n    assert satisfies(\"2.28.9\", \"~=2.28.1\") and not satisfies(\"2.29.0\", \"~=2.28.1\") and not satisfies(\"2.28.0\", \"~=2.28.1\")\n    assert satisfies(\"0.1\", \"\") and satisfies(\"5\", \" > 4 , <= 5 \") and not satisfies(\"4\", \">4\")\n    for spec, message in [(\"~=2\", \"~= needs at least two release numbers\"), (\"=>2\", \"bad specifier: '=>2'\"), (\"2.0\", \"bad specifier: '2.0'\")]:\n        try:\n            satisfies(\"2.0\", spec)\n        except ValueError as e:\n            assert str(e) == message, spec\n        else:\n            raise AssertionError(spec)\n\ndef test_best():\n    \"the newest acceptable final release\"\n    available = [\"2.27.1\", \"2.28.0\", \"2.28.2\", \"2.31.0\", \"2.32.0rc1\", \"3.0.0\"]\n    assert best(available, \">=2.28,<3\") == \"2.31.0\" and best(available, \"~=2.28.0\") == \"2.28.2\"\n    assert best(available, \">=4\") is None and best([\"1.0rc1\"], \"\") is None")

### Exercise: Auditing a lock file

Write `normalise(name)`: the PEP 503 form (lower case, every run of `-`, `_` and `.` replaced by one `-`). Then `audit(locked, advisories)`: `locked` maps package names (any spelling) to version texts; each advisory is a dict with `id`, `package`, `affected` (a list of specifier strings, the version is affected if it satisfies **any** of them), `fixed` (a list of version texts) and `severity` (a CVSS score). Return a list of findings, `{"package": NORMALISED, "version": V, "id": ID, "severity": S, "fix": F}`, where F is the smallest fixed version greater than the installed one, or `None` if there isn't one; sorted by severity (highest first), then package, then id. `satisfies` and `Version` are given.

In [ ]:
import functools
import re

PRE_ORDER = {"a": 0, "b": 1, "rc": 2}

@functools.total_ordering
class Version:
    def __init__(self, text):
        text = text.strip()
        m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", text)
        if not m:
            raise ValueError(f"not a version: {text!r}")
        self.text = text
        self.release = tuple(int(x) for x in m[1].split("."))
        self.pre = (m[2], int(m[3])) if m[2] else None
    def _key(self):
        release = list(self.release)
        while len(release) > 1 and release[-1] == 0:
            release.pop()
        return tuple(release), (PRE_ORDER[self.pre[0]], self.pre[1]) if self.pre else (3, 0)
    def __str__(self):
        return self.text
    def __eq__(self, other):
        return self._key() == other._key()
    def __lt__(self, other):
        return self._key() < other._key()
    def __hash__(self):
        return hash(self._key())

OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, "<=": lambda a, b: a <= b,
       ">=": lambda a, b: a >= b, "<": lambda a, b: a < b, ">": lambda a, b: a > b}

def satisfies(version, spec):
    v = Version(version)
    for part in [p for p in spec.replace(" ", "").split(",") if p]:
        op = next(o for o in ("==", "!=", "<=", ">=", "<", ">") if part.startswith(o))
        if not OPS[op](v, Version(part[len(op):])):
            return False
    return True

def normalise(name):
    ___

def audit(locked, advisories):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ADVISORIES = [\n    {\"id\": \"PYSEC-2023-74\", \"package\": \"requests\", \"affected\": [\">=2.3.0,<2.31.0\"], \"fixed\": [\"2.31.0\"], \"severity\": 6.1},\n    {\"id\": \"GHSA-9wx4\", \"package\": \"Jinja2\", \"affected\": [\"<3.1.3\"], \"fixed\": [\"3.1.3\"], \"severity\": 5.4},\n    {\"id\": \"GHSA-h5c8\", \"package\": \"jinja2\", \"affected\": [\"<3.1.4\"], \"fixed\": [\"3.1.4\"], \"severity\": 5.4},\n    {\"id\": \"CVE-2024-1135\", \"package\": \"gunicorn\", \"affected\": [\"<22.0.0\"], \"fixed\": [\"22.0.0\"], \"severity\": 7.5},\n    {\"id\": \"PYSEC-2021-1\", \"package\": \"PyYAML\", \"affected\": [\"<5.4\", \">=6.0,<6.0.1\"], \"fixed\": [\"5.4\", \"6.0.1\"], \"severity\": 9.8},\n    {\"id\": \"GHSA-old\", \"package\": \"abandoned.lib\", \"affected\": [\">=0\"], \"fixed\": [], \"severity\": 4.0},\n]\n\ndef test_normalise():\n    \"PEP 503\"\n    assert normalise(\"Flask_SQLAlchemy\") == normalise(\"flask.sqlalchemy\") == normalise(\"flask--sqlalchemy\") == \"flask-sqlalchemy\"\n    assert normalise(\"PyYAML\") == \"pyyaml\"\n\ndef test_audit():\n    \"findings, fixes and order\"\n    locked = {\"Requests\": \"2.28.1\", \"jinja2\": \"3.1.2\", \"gunicorn\": \"22.0.0\", \"pyyaml\": \"6.0\", \"Abandoned_Lib\": \"1.2\", \"click\": \"8.1.7\"}\n    got = audit(locked, ADVISORIES)\n    assert got == [\n        {\"package\": \"pyyaml\", \"version\": \"6.0\", \"id\": \"PYSEC-2021-1\", \"severity\": 9.8, \"fix\": \"6.0.1\"},\n        {\"package\": \"requests\", \"version\": \"2.28.1\", \"id\": \"PYSEC-2023-74\", \"severity\": 6.1, \"fix\": \"2.31.0\"},\n        {\"package\": \"jinja2\", \"version\": \"3.1.2\", \"id\": \"GHSA-9wx4\", \"severity\": 5.4, \"fix\": \"3.1.3\"},\n        {\"package\": \"jinja2\", \"version\": \"3.1.2\", \"id\": \"GHSA-h5c8\", \"severity\": 5.4, \"fix\": \"3.1.4\"},\n        {\"package\": \"abandoned-lib\", \"version\": \"1.2\", \"id\": \"GHSA-old\", \"severity\": 4.0, \"fix\": None}]\n    assert audit({\"pyyaml\": \"5.3.1\"}, ADVISORIES)[0][\"fix\"] == \"5.4\" and audit({\"requests\": \"2.32.3\"}, ADVISORIES) == []")

### Exercise: Installing only what was locked

A hashed requirements file has entries like `NAME==VERSION --hash=sha256:HEX` (one or more `--hash` options; real files often split them over lines ending with `\`, which you should join first). Write `parse_locked(text)`: skip blank lines and `#` comments, and return `{normalised_name: (version, {hex, ...})}`; raise `ValueError(f"not pinned: {LINE}")` for an entry without `==` or without any `--hash=sha256:`. Then `verify(locked, name, version, data)`: return `True` only if the package is locked at exactly that version and the SHA-256 of `data` (bytes) is one of its hashes (compare hex digests in lower case); otherwise `False`.

In [ ]:
import hashlib
import re

def parse_locked(text):
    ___

def verify(locked, name, version, data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lock():\n    \"hashes and pins\"\n    import hashlib\n    good = b\"pretend this is a wheel\"\n    h1, h2 = hashlib.sha256(good).hexdigest(), hashlib.sha256(b\"the sdist\").hexdigest()\n    text = f\"\"\"# generated by pip-compile --generate-hashes\nFlask==3.0.3 \\\\\n    --hash=sha256:{h1.upper()} \\\\\n    --hash=sha256:{h2}\n\nclick==8.1.7 --hash=sha256:{\"ab\" * 32}\n\"\"\"\n    locked = parse_locked(text)\n    assert locked == {\"flask\": (\"3.0.3\", {h1, h2}), \"click\": (\"8.1.7\", {\"ab\" * 32})}\n    assert verify(locked, \"flask\", \"3.0.3\", good) and verify(locked, \"Flask\", \"3.0.3\", b\"the sdist\")\n    assert not verify(locked, \"flask\", \"3.0.3\", good + b\"!\") and not verify(locked, \"flask\", \"3.0.4\", good)\n    assert not verify(locked, \"requests\", \"2.32.3\", good)\n    for bad in (\"requests>=2.28 --hash=sha256:\" + \"a\" * 64, \"requests==2.32.3\"):\n        try:\n            parse_locked(bad)\n        except ValueError as e:\n            assert str(e) == f\"not pinned: {bad}\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Why is this installed?

`deps` maps each package to the list of packages it depends on directly (names already normalised). Write `installed(direct, deps)`: every package installed for the list of direct dependencies, transitive ones included, sorted. Then `why(package, direct, deps)`: every chain from a direct dependency down to `package`, as lists of names (`["flask", "werkzeug", "markupsafe"]`), sorted; a direct dependency that **is** the package gives a chain of one. Cycles in `deps` must not loop forever (a chain never visits a package twice).

In [ ]:
def installed(direct, deps):
    ___

def why(package, direct, deps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("DEPS = {\"flask\": [\"werkzeug\", \"jinja2\", \"click\", \"itsdangerous\", \"blinker\"], \"werkzeug\": [\"markupsafe\"], \"jinja2\": [\"markupsafe\"],\n        \"click\": [], \"itsdangerous\": [], \"blinker\": [], \"markupsafe\": [], \"requests\": [\"urllib3\", \"idna\", \"certifi\", \"charset-normalizer\"],\n        \"urllib3\": [], \"idna\": [], \"certifi\": [], \"charset-normalizer\": [], \"a\": [\"b\"], \"b\": [\"a\", \"markupsafe\"]}\n\ndef test_installed():\n    \"the whole tree\"\n    assert installed([\"flask\"], DEPS) == [\"blinker\", \"click\", \"flask\", \"itsdangerous\", \"jinja2\", \"markupsafe\", \"werkzeug\"]\n    assert len(installed([\"flask\", \"requests\"], DEPS)) == 12 and installed([\"a\"], DEPS) == [\"a\", \"b\", \"markupsafe\"]\n\ndef test_why():\n    \"every chain\"\n    assert why(\"markupsafe\", [\"flask\", \"requests\"], DEPS) == [[\"flask\", \"jinja2\", \"markupsafe\"], [\"flask\", \"werkzeug\", \"markupsafe\"]]\n    assert why(\"requests\", [\"flask\", \"requests\"], DEPS) == [[\"requests\"]]\n    assert why(\"markupsafe\", [\"a\"], DEPS) == [[\"a\", \"b\", \"markupsafe\"]] and why(\"idna\", [\"flask\"], DEPS) == []")

### Exercise: Spotting typosquats

Write `distance(a, b)`: the Damerau-Levenshtein distance (the fewest single-character insertions, deletions, substitutions **or swaps of two neighbouring characters** that turn `a` into `b`; the "optimal string alignment" version is fine). Then `suspicious(name, popular)`: compare the normalised name (PEP 503) with each normalised popular name; return the sorted list of popular names it's **not equal** to but looks like: distance 1 for names of up to 8 characters, up to 2 for longer ones, or equal once all `-` are removed (`python-dateutil` versus `pythondateutil`).

In [ ]:
import re

def distance(a, b):
    ___

def suspicious(name, popular):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("POPULAR = [\"requests\", \"numpy\", \"python-dateutil\", \"beautifulsoup4\", \"django\", \"urllib3\", \"pyyaml\"]\n\ndef test_distance():\n    \"edits and swaps\"\n    assert distance(\"requests\", \"requests\") == 0 and distance(\"reqeusts\", \"requests\") == 1\n    assert distance(\"kitten\", \"sitting\") == 3 and distance(\"\", \"abc\") == 3 and distance(\"ab\", \"ba\") == 1\n\ndef test_suspicious():\n    \"look-alikes\"\n    assert suspicious(\"reqeusts\", POPULAR) == [\"requests\"] and suspicious(\"requests\", POPULAR) == []\n    assert suspicious(\"nunpy\", POPULAR) == [\"numpy\"] and suspicious(\"Python_DateUtils\", POPULAR) == [\"python-dateutil\"]\n    assert suspicious(\"pythondateutil\", POPULAR) == [\"python-dateutil\"] and suspicious(\"beautifulsoup\", POPULAR) == [\"beautifulsoup4\"]\n    assert suspicious(\"djangoo\", POPULAR) == [\"django\"] and suspicious(\"flask\", POPULAR) == [] and suspicious(\"PyYAML\", POPULAR) == []\n    assert suspicious(\"urlib3\", POPULAR) == [\"urllib3\"] and suspicious(\"pandas\", POPULAR) == []")

### Bug hunt: An upgrade that looked like a downgrade

The check compares versions as strings, so `"1.10.0" < "1.9.0"` and the fixed version looks **older**: this prints `still vulnerable: True`. Compare numerically, so it prints `still vulnerable: False`.

In [ ]:
installed = "1.10.0"
fixed_in = "1.9.0"
print("still vulnerable:", installed < fixed_in)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tuples of numbers\"\n    expect_output(\"still vulnerable: False\")")

### Bug hunt: An advisory that matched nothing

The advisory names the package `PyYAML`, but the lock file says `pyyaml`, and the audit compares the names exactly, so it finds nothing and prints `0 finding(s)`. Normalise both names (PEP 503), so it prints `1 finding(s)`.

In [ ]:
import re

advisories = [{"package": "PyYAML", "id": "PYSEC-2021-1"}, {"package": "requests", "id": "PYSEC-2023-74"}]
locked = {"pyyaml": "5.3.1", "flask": "3.0.3"}
found = [a for a in advisories if a["package"] in locked]
print(len(found), "finding(s)")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"normalise\"\n    expect_output(\"1 finding(s)\")")

### Bug hunt: A genuine file rejected

The lock file stores hashes in capitals, but `hexdigest()` is lower case, so the comparison fails for the genuine file and this prints `genuine: False, tampered: False`. Compare them case-insensitively, so it prints `genuine: True, tampered: False`.

In [ ]:
import hashlib

wheel = b"genuine wheel bytes"
locked_hash = hashlib.sha256(wheel).hexdigest().upper()

def ok(data):
    return hashlib.sha256(data).hexdigest() == locked_hash

print(f"genuine: {ok(wheel)}, tampered: {ok(wheel + b'!')}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"lower-case both\"\n    expect_output(\"genuine: True, tampered: False\")")

### Puzzle: Adding a dependency carefully

Arrange the steps for adding a new package to a project. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: Version("2.0rc1") < Version("2.0")
q = ___   # the versions ~=2.28 allows, written like "2.*"
r = ___   # the PEP 503 form of "Zope.Interface"
s = ___   # the pip option that refuses files whose hashes don't match
t = ___   # "--index-url" or "--extra-index-url": the safer way to use a private index

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"2.*\" and r == \"zope-interface\", \"p, q and r\"\n    assert s == \"--require-hashes\" and t == \"--index-url\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a dependency report

Write `report(direct, locked_text, advisories, popular, licences, allowed)`. `direct` maps the project's direct dependencies to their specifiers (from `pyproject.toml`); `locked_text` is a hashed lock file (parse it with the given `parse_locked`); `advisories` are as in today's audit (use the given `audit`); `popular` is a list of popular package names; `licences` maps normalised names to licence identifiers; `allowed` is the set of acceptable licences. Return `(lines, code)`: `lines` holds one line per problem, in this order of sections, each section's lines sorted: `missing from lock: NAME` for a direct dependency not in the lock; `outside range: NAME VERSION (wants SPEC)` when the locked version doesn't satisfy the direct dependency's specifier; `vulnerable: NAME VERSION ID (severity S): upgrade to FIX` (or `: no fix yet`), in the audit's order rather than sorted; `licence: NAME uses LICENCE` for a locked package whose licence isn't allowed (or `licence: NAME has no known licence` if it's missing from `licences`); and `possible typosquat: NAME looks like POPULAR` for each locked package that `suspicious` flags (one line per popular name). Names in lines are normalised. `code` is `0` if there are no problems, `2` if any vulnerability has severity 7.0 or more, else `1`. If there are no problems, `lines` is `["no problems found"]`.

In [ ]:
import re

def norm(name):
    return re.sub(r"[-_.]+", "-", name).lower()

def key(text):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", text.strip())
    nums = [int(x) for x in m[1].split(".")]
    while len(nums) > 1 and nums[-1] == 0:
        nums.pop()
    return tuple(nums), ({"a": 0, "b": 1, "rc": 2}[m[2]], int(m[3])) if m[2] else (3, 0)

def satisfies(version, spec):
    ops = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, "<=": lambda a, b: a <= b,
           ">=": lambda a, b: a >= b, "<": lambda a, b: a < b, ">": lambda a, b: a > b}
    for part in [p for p in spec.replace(" ", "").split(",") if p]:
        op = next(o for o in ("==", "!=", "<=", ">=", "<", ">") if part.startswith(o))
        if not ops[op](key(version), key(part[len(op):])):
            return False
    return True

def parse_locked(text):
    locked = {}
    for line in re.sub(r"\\\n", " ", text).splitlines():
        line = " ".join(line.split())
        if line and not line.startswith("#"):
            name, version = line.split()[0].split("==", 1)
            locked[norm(name)] = (version, set(re.findall(r"--hash=sha256:([0-9a-fA-F]{64})", line)))
    return locked

def audit(locked_versions, advisories):
    found = []
    for adv in advisories:
        name = norm(adv["package"])
        version = locked_versions.get(name)
        if version is None or not any(satisfies(version, s) for s in adv["affected"]):
            continue
        later = sorted((f for f in adv["fixed"] if key(f) > key(version)), key=key)
        found.append({"package": name, "version": version, "id": adv["id"], "severity": adv["severity"], "fix": later[0] if later else None})
    return sorted(found, key=lambda f: (-f["severity"], f["package"], f["id"]))

def distance(a, b):
    d = [[i + j if i * j == 0 else 0 for j in range(len(b) + 1)] for i in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (a[i - 1] != b[j - 1]))
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    return d[len(a)][len(b)]

def suspicious(name, popular):
    n = norm(name)
    return sorted(p for p in popular if norm(p) != n and (distance(n, norm(p)) <= (1 if len(norm(p)) <= 8 else 2)
                                                          or n.replace("-", "") == norm(p).replace("-", "")))

def report(direct, locked_text, advisories, popular, licences, allowed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("H = \"--hash=sha256:\" + \"a\" * 64\nLOCK = f\"\"\"requests==2.28.1 {H}\nurllib3==1.26.18 {H}\nFlask==3.0.3 {H}\njinja2==3.1.2 {H}\nreqeusts==1.0 {H}\ngpl-thing==0.3 {H}\n\"\"\"\nADVISORIES = [\n    {\"id\": \"PYSEC-2023-74\", \"package\": \"requests\", \"affected\": [\">=2.3.0,<2.31.0\"], \"fixed\": [\"2.31.0\"], \"severity\": 6.1},\n    {\"id\": \"GHSA-9wx4\", \"package\": \"Jinja2\", \"affected\": [\"<3.1.3\"], \"fixed\": [\"3.1.3\"], \"severity\": 5.4},\n    {\"id\": \"CVE-2024-37891\", \"package\": \"urllib3\", \"affected\": [\"<1.26.19\", \">=2.0.0,<2.2.2\"], \"fixed\": [\"1.26.19\", \"2.2.2\"], \"severity\": 7.2},\n    {\"id\": \"GHSA-none\", \"package\": \"gpl_thing\", \"affected\": [\">=0\"], \"fixed\": [], \"severity\": 3.0},\n]\nLICENCES = {\"requests\": \"Apache-2.0\", \"urllib3\": \"MIT\", \"flask\": \"BSD-3-Clause\", \"jinja2\": \"BSD-3-Clause\", \"gpl-thing\": \"GPL-3.0\"}\nALLOWED = {\"MIT\", \"BSD-3-Clause\", \"Apache-2.0\"}\nPOPULAR = [\"requests\", \"flask\", \"jinja2\", \"urllib3\"]\n\ndef test_report():\n    \"every section\"\n    direct = {\"requests\": \">=2.31,<3\", \"flask\": \">=3\", \"click\": \">=8\", \"gpl_thing\": \"\"}\n    lines, code = report(direct, LOCK, ADVISORIES, POPULAR, LICENCES, ALLOWED)\n    assert lines == [\n        \"missing from lock: click\",\n        \"outside range: requests 2.28.1 (wants >=2.31,<3)\",\n        \"vulnerable: urllib3 1.26.18 CVE-2024-37891 (severity 7.2): upgrade to 1.26.19\",\n        \"vulnerable: requests 2.28.1 PYSEC-2023-74 (severity 6.1): upgrade to 2.31.0\",\n        \"vulnerable: jinja2 3.1.2 GHSA-9wx4 (severity 5.4): upgrade to 3.1.3\",\n        \"vulnerable: gpl-thing 0.3 GHSA-none (severity 3.0): no fix yet\",\n        \"licence: gpl-thing uses GPL-3.0\",\n        \"licence: reqeusts has no known licence\",\n        \"possible typosquat: reqeusts looks like requests\"]\n    assert code == 2\n\ndef test_clean_and_codes():\n    \"exit codes\"\n    lock = f\"flask==3.0.3 {H}\\njinja2==3.1.4 {H}\\n\"\n    assert report({\"Flask\": \">=3\"}, lock, ADVISORIES, POPULAR, LICENCES, ALLOWED) == ([\"no problems found\"], 0)\n    lines, code = report({\"flask\": \">=3\"}, f\"flask==3.0.3 {H}\\njinja2==3.1.2 {H}\\n\", ADVISORIES, POPULAR, LICENCES, ALLOWED)\n    assert lines == [\"vulnerable: jinja2 3.1.2 GHSA-9wx4 (severity 5.4): upgrade to 3.1.3\"] and code == 1")

## Warm-up

Version numbers aren't text: as strings, `"1.10.0"` sorts before `"1.9.0"`, because `"1"` comes before `"9"`. A tool that compared versions as strings would think a vulnerable `1.9.0` was newer than the fixed `1.10.0`. Compare them as tuples of numbers, and handle the rest of the rules, as pip's `packaging` library does.

## The supply chain

Your **dependencies** are the packages you install directly; **transitive** dependencies are theirs, and theirs. A typical Flask app installs about ten packages for the one you asked for; a data-science project, hundreds. Every one is code that runs with your program's permissions, and some run code **while they install**. The main risks:

- **Known vulnerabilities**: a bug in a version you use, published as an advisory with an identifier (a **CVE** such as `CVE-2023-32681`, or a `GHSA-...` or `PYSEC-...` id) and the range of affected versions. Attackers read advisories too, so old versions in production are the easiest way in.
- **Typosquatting**: a malicious package named like a popular one (`reqeusts`, `python-dateutils`, `beautifulsoup` instead of `beautifulsoup4`) waiting for a typing mistake. Hundreds are found on PyPI every year.
- **Dependency confusion**: your company's internal package `acme-billing` lives on a private index; an attacker publishes `acme-billing` with a higher version number on the **public** index, and an installer that looks at both picks theirs.
- **Compromised maintainers or updates**: a stolen account, or a hostile new owner, publishes a malicious **new version** of a trusted package; everyone who installs "the latest" gets it.
- **Abandoned packages**: no fixes coming, ever.
- **Licences**: some licences place conditions on how you distribute your own code (Week 73).

The defences fit together: **know** what you run (a lock file, or an SBOM: a "software bill of materials" listing every component and version), **pin** it exactly with hashes so nothing changes under you, **audit** it against advisories continuously, **update** deliberately and often, and **think** before adding anything new.

## Versions and specifiers

Python versions follow **PEP 440**: a release made of numbers separated by dots (`2.31.0`), optionally followed by a pre-release marker: `a` (alpha), `b` (beta) or `rc` (release candidate) and a number (`3.0rc1`). Ordering rules:

- compare the release numbers as a tuple, after removing trailing zeros, so `1.0` equals `1.0.0`, and `1.10` is after `1.9`;
- for the same release, pre-releases come **before** the final: `2.0a1 < 2.0b2 < 2.0rc1 < 2.0`.

(The full standard also has post-releases, development releases, local versions and epochs; `packaging.version.Version` handles them all.)

A **specifier** says which versions are acceptable, as a comma-separated list of conditions that must **all** hold:

| Specifier | Means |
|---|---|
| `==2.31.0` | exactly that version |
| `!=2.30.0` | anything but that |
| `>=2.28`, `<3` | the usual comparisons |
| `~=2.28` | "compatible release": `>=2.28` and `==2.*` (the last number may rise) |
| `~=2.28.1` | `>=2.28.1` and `==2.28.*` |

`requests>=2.28,<3` in your `pyproject.toml` (Week 73) says which versions your code works with; a **lock file** records the exact versions actually chosen.

## Locking and auditing

**Lock exactly.** A lock file lists **every** package, transitive ones included, at an exact version, with the **hashes** of the files that may be installed:

```
requests==2.32.3 \
    --hash=sha256:70761cfe03c773ceb22aa2f671b4757976145175cdfca038c02654d061d6dcc6
```

Installing with `pip install --require-hashes -r requirements.txt` refuses any file whose SHA-256 doesn't match, so a package replaced on the index (or a man-in-the-middle) can't slip in. Tools write lock files for you: `pip-compile --generate-hashes` (pip-tools), `uv lock`, `poetry lock`. Commit the lock file, and install from it everywhere (CI, containers, production), so what you tested is what runs.

**Audit continuously.** An audit takes the locked versions and checks each against a database of advisories (the PyPA Advisory Database, OSV, GitHub's advisories). Each advisory lists the **affected** versions (usually as ranges) and the **fixed** ones. For each match, you want the smallest version that fixes it (the least disruptive upgrade), and a severity (often a **CVSS** score from 0 to 10) to decide what to do first. Not every advisory matters to you (the vulnerable function may be one you never call), but checking that is work; upgrading is usually quicker.

Package names need **normalising** before you compare them: PyPI treats `PyYAML`, `pyyaml` and `py_yaml` as... not quite: it lower-cases and treats runs of `-`, `_` and `.` as one `-` (PEP 503), so `Flask_SQLAlchemy`, `flask.sqlalchemy` and `flask-sqlalchemy` are the same project.

**Update deliberately.** Small, frequent updates are easier than rare huge ones. Bots such as **Dependabot** and **Renovate** open a pull request when a new version (or a security fix) appears; your CI (Week 74) runs the tests on it, and you merge when it's green.

## Choosing dependencies

Before adding a package, spend five minutes on it:

- **Is it really needed?** Twenty lines of your own code can be better than a dependency with its own dependencies. The standard library covers a lot.
- **Is it the real one?** Check the exact name on PyPI and in the project's documentation; look for look-alikes.
- **Is it alive and trusted?** Recent releases, several maintainers, an active repository, a security policy, many dependents.
- **What does it bring with it?** Its own dependency tree, and anything it runs at install time.
- **Is its licence acceptable** for how you'll distribute your code?

And protect the installer itself: install internal packages only from your own index (`--index-url`, not `--extra-index-url`, which mixes indexes and invites dependency confusion), and register your internal names on the public index too, so nobody else can.

### On your own computer

`pip-audit` checks an environment or a requirements file against advisories (`pip install pip-audit`, then `pip-audit -r requirements.txt`); `pip list --outdated` shows what's behind; `pipdeptree` shows why each package is installed; `uv lock` and `pip-compile --generate-hashes` write hashed lock files; and GitHub's Dependabot can be switched on in a repository's settings. Publishers can use PyPI's **Trusted Publishing**, which publishes from CI without long-lived passwords and attaches provenance, so users can check where a release was built.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run `pip-audit` and `pip list --outdated` in a project's virtual environment, and look up one finding's advisory to see what the vulnerability actually was. Tomorrow: **cryptography basics**: hashing, message authentication, password storage, encryption and signatures, and which of them to reach for when.